# Vue2 + Vue3 框架 · 第 02 天

## 指令修饰符、样式绑定、表单增强、计算属性与侦听器

> 本篇依据第 02 天的 MD 笔记、PPT 和随堂代码整理。当天的示例主要采用 **Vue 2 Options API**（`new Vue`、`data`、`computed`、`watch`），这些概念在 Vue 3 的 Options API 中仍然适用；项目开发时也会逐步过渡到 Vue 3 Composition API。

第 01 天解决了“数据如何渲染到页面”的问题。本天继续解决四类实际需求：

1. 用修饰符减少事件和表单的重复代码。
2. 用 `v-bind` 动态控制 `class`、`style` 和页面高亮效果。
3. 用 `computed` 得到基于已有数据的派生结果。
4. 用 `watch` 在数据变化时触发异步请求、持久化等副作用。

## 一、学习目标与知识地图

| 模块 | 关键语法 | 适合解决的问题 |
| --- | --- | --- |
| 指令修饰符 | `.enter`、`.trim`、`.number`、`.stop`、`.prevent` | 键盘回车、输入清理、类型转换、事件冒泡和默认行为 |
| `v-bind` 样式增强 | `:class` 对象/数组、`:style` 对象 | 高亮、禁用、进度条、动态主题 |
| `v-model` 表单 | `input`、`checkbox`、`radio`、`select`、`textarea` | 获取和设置用户输入 |
| 计算属性 | `computed`、`get`、`set` | 依赖数据计算总数、总价、平均值等派生值 |
| 侦听器 | `watch`、`deep`、`immediate` | 翻译请求、防抖、保存本地数据、监视对象 |
| 综合案例 | 成绩管理、购物车 | 把渲染、增删改、统计和持久化串起来 |

### 先记住一个选择原则

- **模板中需要展示一个由多个数据计算出来的结果**：优先使用 `computed`。
- **数据变化后需要执行请求、定时器、写入 `localStorage` 等副作用**：使用 `watch`。
- **一次点击或一次输入只触发一个动作**：使用 `methods`。

## 二、指令修饰符

### 2.1 修饰符是什么

指令修饰符使用点号 `.` 表示，是 Vue 为常见操作封装的后缀。它让模板表达意图，减少手写原生事件对象处理的代码。

```html
<!-- 键盘修饰符 -->
<input @keyup.enter="search">

<!-- 表单修饰符 -->
<input v-model.trim="username">
<input v-model.number="age">

<!-- 事件修饰符 -->
<button @click.stop="save">保存</button>
<a href="/detail" @click.prevent="openDetail">详情</a>
```

### 2.2 常用修饰符

| 修饰符 | 用法 | 作用 |
| --- | --- | --- |
| `.enter` | `@keyup.enter` | 只有按下 Enter 才触发处理函数 |
| `.trim` | `v-model.trim` | 去掉输入值首尾空格 |
| `.number` | `v-model.number` | 尝试把输入值转换成数字 |
| `.stop` | `@click.stop` | 调用 `stopPropagation()`，阻止冒泡 |
| `.prevent` | `@submit.prevent` | 调用 `preventDefault()`，阻止默认行为 |

`.stop` 和 `.prevent` 可以连用，例如 `@click.stop.prevent`。修饰符的顺序也会影响生成的处理逻辑，书写时按照“先阻止什么，再执行什么”的意图排列。

### 2.3 修饰符综合示例

```html
<div id="app">
  <h3>回车搜索与表单修饰符</h3>
  <form @submit.prevent="search">
    <input v-model.trim="keyword" @keyup.enter="search" placeholder="输入关键词后回车">
    <input v-model.number="pageSize" type="number" min="1">
    <button>搜索</button>
  </form>
  <p>关键词：{{ keyword || '（空）' }}</p>
  <p>每页数量：{{ pageSize }}，类型：{{ typeof pageSize }}</p>
  <p v-if="message">{{ message }}</p>
</div>

<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script>
new Vue({
  el: '#app',
  data: {
    keyword: '',
    pageSize: 10,
    message: ''
  },
  methods: {
    search () {
      if (!this.keyword) {
        this.message = '请输入关键词'
        return
      }
      this.message = `正在搜索：${this.keyword}，每页 ${this.pageSize} 条`
    }
  }
})
</script>
```

注意：`v-model.number` 对空字符串、无法转换的文本会保留字符串值；不要只看输入框外观，应在业务校验时确认类型。

## 三、`v-bind` 对样式的增强

### 3.1 动态绑定 `class`

静态类名和动态类名可以同时存在：

```html
<div class="box" :class="{ pink: isPink, big: isBig }">内容</div>
```

#### 对象语法

对象的 **键是类名，值是布尔表达式**。值为真时添加该类，值为假时移除该类，适合“某个状态来回切换”的场景。

```html
<div :class="{ active: currentIndex === index, disabled: item.disabled }">
  {{ item.name }}
</div>
```

#### 数组语法

数组中的类名都会被添加，适合一次批量添加类名，也可以混合变量和三元表达式：

```html
<div :class="['box', themeClass, isBig ? 'big' : 'small']"></div>
```

常见易错点：类名本身带短横线时可以写成字符串；对象键不能直接写未经引号包裹的特殊表达式。

### 3.2 京东秒杀 Tab 切换

实现步骤：

1. 把页签名称放进数组，用 `v-for` 渲染。
2. 用 `activeIndex` 记录当前高亮的索引。
3. 点击页签时更新索引。
4. 使用 `:class` 为当前项添加 `active` 类。

```html
<style>
  ul { display: flex; padding: 0; border-bottom: 2px solid #e01222; }
  li { list-style: none; width: 110px; text-align: center; }
  a { display: block; padding: 14px 8px; color: #333; text-decoration: none; }
  a.active { color: #fff; background: #e01222; }
</style>

<div id="app">
  <ul>
    <li v-for="(item, index) in tabs" :key="item.id">
      <a href="#" :class="{ active: activeIndex === index }" @click.prevent="activeIndex = index">
        {{ item.name }}
      </a>
    </li>
  </ul>
  <p>当前页签：{{ tabs[activeIndex].name }}</p>
</div>

<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script>
new Vue({
  el: '#app',
  data: {
    activeIndex: 0,
    tabs: [
      { id: 1, name: '京东秒杀' },
      { id: 2, name: '每日特价' },
      { id: 3, name: '品类秒杀' }
    ]
  }
})
</script>
```

### 3.3 动态绑定 `style`

对象语法：

```html
<div class="box" :style="{ backgroundColor: color, fontSize: size + 'px' }">
  动态样式
</div>
```

CSS 属性名可以使用驼峰写法（`backgroundColor`），也可以使用带引号的短横线写法（`'background-color'`）。数组语法可以合并多个样式对象：

```html
<div :style="[baseStyle, activeStyle]"></div>
```

### 3.4 进度条案例

```html
<style>
  .progress { width: 400px; height: 24px; border: 3px solid #272425; border-radius: 15px; }
  .inner { height: 100%; border-radius: 10px; background: #409eff; transition: width .4s; }
</style>

<div id="app">
  <div class="progress">
    <div class="inner" :style="{ width: percent + '%' }"></div>
  </div>
  <p>当前进度：{{ percent }}%</p>
  <button v-for="value in [25, 50, 75, 100]" :key="value" @click="percent = value">
    设置 {{ value }}%
  </button>
</div>

<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script>
new Vue({ el: '#app', data: { percent: 50 } })
</script>
```

## 四、`v-model` 在其他表单元素中的使用

`v-model` 会根据控件类型选择合适的属性和事件：

| 控件 | 双向绑定的值 | 典型结果 |
| --- | --- | --- |
| `input:text`、`textarea` | `value` | 字符串 |
| 单个 `checkbox` | `checked` | 布尔值 |
| 多个 `checkbox` | `checked` + `value` | 数组 |
| `radio` | `checked` + `value` | 被选中的字符串 |
| `select` | `value` | 选项的 `value` |

### 4.1 完整表单示例

```html
<div id="app">
  <h3>小黑学习网报名表</h3>
  <label>姓名：<input v-model.trim="form.name"></label><br><br>

  <label>
    <input type="checkbox" v-model="form.single">
    是否单身
  </label><br><br>

  性别：
  <label><input type="radio" value="男" v-model="form.gender">男</label>
  <label><input type="radio" value="女" v-model="form.gender">女</label><br><br>

  城市：
  <select v-model="form.city">
    <option value="北京">北京</option>
    <option value="上海">上海</option>
    <option value="成都">成都</option>
  </select><br><br>

  技能：
  <label v-for="skill in skills" :key="skill">
    <input type="checkbox" :value="skill" v-model="form.skills">{{ skill }}
  </label><br><br>

  自我描述：<textarea v-model="form.intro"></textarea><br>
  <button @click="submit">立即注册</button>

  <pre>{{ form }}</pre>
</div>

<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script>
new Vue({
  el: '#app',
  data: {
    skills: ['HTML/CSS', 'JavaScript', 'Vue'],
    form: { name: '', single: false, gender: '男', city: '北京', skills: [], intro: '' }
  },
  methods: {
    submit () {
      if (!this.form.name) return alert('请填写姓名')
      console.log('提交数据：', this.form)
    }
  }
})
</script>
```

单选框和复选框一定要设置 `value`，否则提交到业务层的数据无法区分具体选项。多个复选框绑定同一个数组，Vue 会自动追加或移除对应的 `value`。

## 五、计算属性 `computed`

### 5.1 概念和基本语法

计算属性是“根据已有数据计算出的新属性”。它的结果由依赖项决定，依赖项变化时自动重新计算；在依赖项没有变化时，Vue 会复用缓存的结果。

```js
new Vue({
  data: {
    list: [
      { id: 1, name: '篮球', num: 1 },
      { id: 2, name: '玩具', num: 2 },
      { id: 3, name: '铅笔', num: 5 }
    ]
  },
  computed: {
    totalCount () {
      return this.list.reduce((sum, item) => sum + item.num, 0)
    }
  }
})
```

模板中像普通属性一样使用：`{{ totalCount }}`。在 JavaScript 中使用 `this.totalCount`，不要写成 `this.totalCount()`。

### 5.2 计算属性的规则

- `computed` 与 `data`、`methods` 同级。
- 每个计算属性对应一个函数，但使用时是属性。
- 计算属性不能与 `data` 中的属性同名。
- 函数中的 `this` 指向 Vue 实例。
- 一个结果依赖多个字段时，计算属性比模板中堆叠表达式更清晰。

### 5.3 `computed` 与 `methods` 的区别

| 对比项 | `computed` | `methods` |
| --- | --- | --- |
| 调用方式 | `totalCount` | `getTotal()` |
| 是否缓存 | 有缓存，依赖不变时复用 | 每次调用都会执行 |
| 适用场景 | 求和、过滤、格式化等派生值 | 点击处理、提交、删除等动作 |
| 是否推荐放副作用 | 不推荐 | 可以执行副作用 |

```html
<p>总数：{{ totalCount }}</p>
<p>总数：{{ getTotal() }}</p>
```

两种写法都可以得到结果，但 `getTotal()` 每次渲染都会执行。只要结果依赖数据且需要展示，就优先考虑计算属性。

### 5.4 计算属性完整写法：`get` 与 `set`

计算属性的简写只提供读取能力：

```js
computed: {
  fullName () {
    return this.firstName + this.lastName
  }
}
```

需要给计算属性赋值时，使用完整写法。`set` 中应该修改真正的数据源，避免直接修改一个临时结果：

```html
<div id="app">
  姓：<input v-model="firstName">
  名：<input v-model="lastName">
  <p>全名：{{ fullName }}</p>
  <button @click="fullName = '诸葛亮'">改名卡</button>
</div>

<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script>
new Vue({
  el: '#app',
  data: { firstName: '刘', lastName: '备' },
  computed: {
    fullName: {
      get () {
        return this.firstName + this.lastName
      },
      set (value) {
        this.firstName = value.slice(0, 1)
        this.lastName = value.slice(1)
      }
    }
  }
})
</script>
```

## 六、侦听器 `watch`

### 6.1 作用

`watch` 用来监视数据变化，并在变化发生后执行一段业务逻辑或异步操作。典型场景包括：

- 输入文字后请求翻译或搜索接口。
- 表单变化后保存到本地存储。
- 监听路由参数后重新拉取数据。
- 数据变化后启动或清理定时器。

不要用 `watch` 代替所有计算逻辑：需要返回一个派生值时使用 `computed`；需要产生副作用时使用 `watch`。

### 6.2 简单写法

```js
watch: {
  // 监视简单类型
  words (newValue, oldValue) {
    console.log('words:', oldValue, '=>', newValue)
  },

  // 监视对象中的指定属性
  'form.city' (newValue) {
    console.log('城市改为：', newValue)
  }
}
```

参数顺序固定为 `newValue, oldValue`。不需要旧值时可以只接收第一个参数。

### 6.3 完整写法：`deep` 与 `immediate`

```js
watch: {
  form: {
    deep: true,       // 监听对象内部的嵌套字段
    immediate: true,  // 创建实例后立即执行一次 handler
    handler (newValue, oldValue) {
      console.log('表单发生变化：', newValue)
    }
  }
}
```

- `deep: true`：对象或数组内部字段变化时也触发。深度监听成本更高，只在确实需要时使用。
- `immediate: true`：初始化时立即执行，适合页面进入时就要请求一次的场景。
- 监听对象时，回调中的 `newValue` 和 `oldValue` 可能引用同一个对象，比较对象内部变化时不要简单依赖全等判断。

### 6.4 翻译案例：深度监听 + 防抖 + 异步请求

课程中的翻译案例同时监听输入内容和语言选择。用户连续输入时，先等待 300ms，避免每敲一个字都发送请求。

```html
<div id="app">
  <select v-model="form.lang">
    <option value="italy">意大利语</option>
    <option value="english">英语</option>
    <option value="german">德语</option>
  </select>
  <textarea v-model="form.words" placeholder="输入要翻译的内容"></textarea>
  <p>翻译结果：{{ result || '等待输入...' }}</p>
</div>

<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script src="https://cdn.jsdelivr.net/npm/axios/dist/axios.min.js"></script>
<script>
new Vue({
  el: '#app',
  data: {
    form: { words: '小黑', lang: 'italy' },
    result: '',
    timer: null
  },
  watch: {
    form: {
      deep: true,
      immediate: true,
      handler (newValue) {
        clearTimeout(this.timer)
        this.timer = setTimeout(async () => {
          if (!newValue.words.trim()) {
            this.result = ''
            return
          }
          try {
            const res = await axios({
              url: 'https://applet-base-api-t.itheima.net/api/translate',
              params: newValue
            })
            this.result = res.data.data
          } catch (error) {
            this.result = '翻译失败，请稍后重试'
            console.error(error)
          }
        }, 300)
      }
    }
  },
  beforeDestroy () {
    clearTimeout(this.timer)
  }
})
</script>
```

这里的 `watch` 做了三件事：先清理上一次定时器，再延迟发请求，最后把响应写入 `result`。请求属于副作用，不应放在计算属性中。

## 七、综合案例一：成绩管理

### 7.1 需求拆分

1. 用 `v-for` 和 `:key` 渲染成绩列表。
2. 点击删除时移除当前行。
3. 使用 `v-model.trim.number` 获取姓名和成绩，并校验空值。
4. 用 `computed` 统计总分和平均分。
5. 使用 `:class` 标记不及格成绩。

### 7.2 可运行实现

```html
<style>
  table { width: 560px; border-collapse: collapse; text-align: center; }
  th, td { border: 1px solid #ddd; padding: 8px; }
  .danger { color: #e34d59; }
  form { margin-bottom: 16px; }
</style>

<div id="app">
  <h2>成绩管理</h2>
  <form @submit.prevent="add">
    <input v-model.trim="form.name" placeholder="姓名">
    <input v-model.number="form.score" type="number" min="0" max="100" placeholder="成绩">
    <button>添加</button>
  </form>

  <table>
    <thead>
      <tr><th>姓名</th><th>成绩</th><th>操作</th></tr>
    </thead>
    <tbody>
      <tr v-for="item in list" :key="item.id">
        <td>{{ item.name }}</td>
        <td :class="{ danger: item.score < 60 }">{{ item.score }}</td>
        <td><a href="#" @click.prevent="remove(item.id)">删除</a></td>
      </tr>
    </tbody>
  </table>

  <p>总分：{{ totalScore }}，平均分：{{ averageScore }}</p>
</div>

<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script>
new Vue({
  el: '#app',
  data: {
    nextId: 4,
    form: { name: '', score: null },
    list: [
      { id: 1, name: '小明', score: 88 },
      { id: 2, name: '小红', score: 59 },
      { id: 3, name: '小刚', score: 76 }
    ]
  },
  computed: {
    totalScore () {
      return this.list.reduce((sum, item) => sum + Number(item.score), 0)
    },
    averageScore () {
      return this.list.length ? (this.totalScore / this.list.length).toFixed(2) : 0
    }
  },
  methods: {
    add () {
      if (!this.form.name || this.form.score === null || this.form.score === '') {
        return alert('请填写完整信息')
      }
      this.list.push({ id: this.nextId++, name: this.form.name, score: this.form.score })
      this.form.name = ''
      this.form.score = null
    },
    remove (id) {
      this.list = this.list.filter(item => item.id !== id)
    }
  }
})
</script>
```

这个案例体现了本天的组合：`v-for` 负责列表、`v-bind` 负责样式、`v-on` 负责动作、`v-model` 负责表单、`computed` 负责统计。

## 八、综合案例二：购物车设计

### 8.1 功能清单

- 商品列表渲染。
- 删除商品。
- 增加或减少数量。
- 全选与反选。
- 计算选中商品的总数量和总价。
- 使用 `watch` 持久化到 `localStorage`。

### 8.2 关键设计

1. 每个商品有唯一 `id`、价格 `price`、数量 `count` 和选中状态 `isChecked`。
2. 全选状态使用计算属性：只有 `every(item => item.isChecked)` 为真时才全选。
3. 全选框的 `set` 遍历所有商品，统一修改 `isChecked`。
4. 总价和总数量只统计已选中的商品。
5. 监听商品数组时开启 `deep: true`，把变化后的数组序列化保存。

### 8.3 关键代码

```js
new Vue({
  el: '#app',
  data: {
    list: JSON.parse(localStorage.getItem('cart-list') || 'null') || [
      { id: 1, name: '苹果', price: 6, count: 2, isChecked: true },
      { id: 2, name: '香蕉', price: 4, count: 3, isChecked: false }
    ]
  },
  computed: {
    isAllChecked: {
      get () {
        return this.list.length > 0 && this.list.every(item => item.isChecked)
      },
      set (value) {
        this.list.forEach(item => { item.isChecked = value })
      }
    },
    checkedCount () {
      return this.list
        .filter(item => item.isChecked)
        .reduce((sum, item) => sum + item.count, 0)
    },
    checkedTotal () {
      return this.list
        .filter(item => item.isChecked)
        .reduce((sum, item) => sum + item.price * item.count, 0)
        .toFixed(2)
    }
  },
  methods: {
    changeCount (item, step) {
      item.count = Math.max(1, item.count + step)
    },
    remove (id) {
      this.list = this.list.filter(item => item.id !== id)
    }
  },
  watch: {
    list: {
      deep: true,
      handler (value) {
        localStorage.setItem('cart-list', JSON.stringify(value))
      }
    }
  }
})
```

`isAllChecked` 使用计算属性完整写法，是因为它既要读取当前是否全选，也要在用户点击全选框时批量写回每一项。

## 九、常见错误与排查顺序

### 9.1 把计算属性当方法调用

```html
<!-- 正确：计算属性像普通属性 -->
{{ totalPrice }}

<!-- 错误：计算属性没有小括号 -->
{{ totalPrice() }}
```

### 9.2 把副作用写进计算属性

计算属性应该返回结果。请求、定时器、`localStorage` 写入等副作用放进 `watch` 或 `methods`。

### 9.3 对象监听不生效

对象内部字段发生变化但没有触发回调时，检查是否需要：

```js
watch: {
  form: {
    deep: true,
    handler () {}
  }
}
```

如果只关心一个字段，优先监听 `'form.words'`，这样范围更小、成本更低。

### 9.4 `v-model.number` 仍然得到字符串

空输入和无法转换的内容可能仍然是字符串。提交前做显式校验：

```js
const score = Number(this.form.score)
if (!Number.isFinite(score)) return
```

### 9.5 全选状态与列表不同步

全选不要只维护一个孤立布尔值，应根据列表的 `isChecked` 计算；删除最后一项时也要考虑空数组。

### 9.6 没有设置 `:key`

`v-for` 渲染列表时使用稳定、唯一的业务 id：

```html
<li v-for="item in list" :key="item.id">{{ item.name }}</li>
```

不要把会变化的数组索引当作默认 key。

## 十、练习题

### 练习 1：Tab 组件

把 Tab 案例扩展为：每个页签有一个内容区域，点击页签时同时切换高亮和内容。要求使用 `v-for`、`:class`、`v-if` 或数组索引完成。

### 练习 2：动态主题

准备 `theme = 'light'` 和 `fontSize = 16`，用 `:class` 切换浅色/深色主题，用 `:style` 动态设置字号，并提供按钮修改这两个数据。

### 练习 3：成绩管理增强

为成绩列表添加“只看不及格”复选框。请用计算属性返回过滤后的列表，不要在模板中直接写复杂的 `filter` 表达式。

### 练习 4：搜索防抖

用 `watch` 监听搜索框，延迟 300ms 打印搜索关键词；输入新内容时清理旧的定时器。输入为空时清空结果。

### 练习 5：购物车持久化

在购物车案例中增加“清空购物车”按钮，并在 `list` 为空时显示空状态文案。

## 十一、当天速记

- 修饰符是指令后缀：键盘用 `.enter`，输入清理用 `.trim`，类型转换用 `.number`，事件控制用 `.stop` 和 `.prevent`。
- `:class` 对象语法适合状态切换，数组语法适合批量添加类名。
- `:style` 绑定对象时，CSS 属性名使用驼峰或带引号的短横线写法。
- `v-model` 会根据控件类型绑定 `value`、`checked` 或选中的 `value`。
- `computed` 是有缓存的派生属性；`methods` 是每次调用都执行的动作函数。
- `computed` 完整写法通过 `get` 读取、`set` 写回源数据。
- `watch` 负责数据变化后的副作用；监听嵌套对象时按需使用 `deep`，初始化就要执行时使用 `immediate`。
- 列表、全选、总价和持久化可以用 `v-for`、`computed`、`watch` 组合实现。